# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassanKh4lil/ML-WEEK-1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Scoring (ranking-flavored), with an optional classification layer.**

My lane is Lane 4 — CTR/Engagement Opportunity Scoring. The core output is a **score per page**: how far a page's CTR sits below what's expected for pages at its position tier and content type. That's a scoring problem, not a hard classification, because "opportunity" is a matter of degree, not a clean yes/no — a page can be a little below expectation or a lot below it, and an editor with limited time wants the list **ordered**, not just split into two buckets.

I'll also define an optional binary proxy on top of the score (`is_ctr_opportunity`, e.g. "in the bottom 20% of its tier with enough volume") so I can evaluate the ranking with classification-style metrics like precision@K — but the underlying task stays a scoring/ranking task, and I'll keep the continuous residual score as the primary output, since collapsing it to yes/no early would throw away exactly the ordering information the editor needs.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `ctr_gap` — a page's CTR minus the expected CTR for its position tier (and later, content type).**

This is a **derived measurement**, not an observed future outcome and not a product decision flag. It's computed directly from two observable columns already in the starter data (`ctr`, `position_tier`), so there's nothing hidden or circular about it — I'm not reusing a FlyRank product score (`health_score`, `priority_score`, etc.), and I'm not touching `trend_direction`/`trend_pct`, which the data dictionary flags as label-only fields I must never use as features.

I'm treating this as a **defined proxy**, not a true observed outcome, and I'll say so plainly in every write-up: `ctr_gap` measures "underperforming its peers right now," not "will get better if I fix the title." A stronger, more honest version for later weeks would be a **future-window** proxy built from the warehouse's daily facts — e.g. does CTR rise in the 30 days *after* a title/meta change — but that requires change-event data I don't have access to yet from the starter CSV alone, so I'm starting here and will revisit once I'm in the warehouse release.

If I add the optional binary layer (`is_ctr_opportunity`), that label is *derived from the same rule* I define for `ctr_gap` — so I'll be explicit that reproducing my own rule with a classifier is a "does this hold up when I test it" check, not a discovery.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@K on the ranked opportunity list**, specifically Precision@50 — mirroring the metric already reported for the starter pipeline in the lane guide (`outputs/model_report.md`), so my number is comparable to a known baseline.

Concretely: of the top 50 pages my score ranks highest, what fraction meet a held-out definition of "real opportunity" (e.g. CTR later measured against a fresh sample, or — more honestly — a same-window check I didn't use to build the score, like "still below expectation after I exclude low-volume noise")? "Good" means clearly beating the lane guide's baseline-rule precision@50 of **0.240** (the starter's rule-only baseline) — anything meaningfully above that is a sign the tier-adjustment is adding real information, not noise.

I'm also going to report **average residual gap size** among the top 50 (how far below expectation they actually sit, in CTR points) as a secondary, plain-language number, since "precision" alone doesn't tell an editor *how much* upside a page carries — only whether it counted as a hit.

I will not use raw accuracy or ROC-AUC as my headline number: the population isn't remotely balanced (most pages aren't real opportunities), and an editor cares about "are my top picks worth reviewing," which is exactly what precision@K measures and accuracy does not.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (page), scored on its trailing-90-day metrics.** The cell below loads the starter CSV, filters to the lane's relevant slice (pages with real position data and a minimum volume so the CTR isn't pure noise), and shows the actual dataframe — including a first sketch of the `ctr_gap` target column.


In [4]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Lane 4 slice: pages that are actually visible and have enough volume to trust their CTR.
# avg_position == 0 means "no data" (per the data dictionary), so exclude those.
lane_df = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 100)].copy()

print(f"full starter set: {len(df):,} rows")
print(f"lane 4 slice (avg_position>0, impressions_90d>=100): {len(lane_df):,} rows, "
      f"{lane_df['client_id'].nunique()} clients")

# unit of analysis: one row = one content item
unit_cols = ["content_id", "client_id", "position_tier", "avg_position",
             "impressions_90d", "clicks_90d", "ctr"]
lane_df[unit_cols].head(8)


full starter set: 30,000 rows
lane 4 slice (avg_position>0, impressions_90d>=100): 22,006 rows, 30 clients


,content_id,client_id,position_tier,avg_position,impressions_90d,clicks_90d,ctr
0,content_304f48230142,client_f369cb89fc,striking,10.6,3803,29,0.76
1,content_a1fb4e703a9e,client_4e07408562,page_3_5,20.3,15320,7,0.05
2,content_9aa793d4d895,client_7f2253d7e2,page_3_5,36.5,12581,11,0.09
3,content_331d6c4de07b,client_19581e27de,page_1,6.2,11751,58,0.49
4,content_d99b7a2d90ca,client_3fdba35f04,page_3_5,44.0,19140,24,0.13
5,content_d4084a4bc775,client_f369cb89fc,page_1,8.5,3970,1,0.03
7,content_a63219c6e95a,client_19581e27de,page_3_5,21.2,1724,1,0.06
8,content_5e6c160719bc,client_6208ef0f77,page_3_5,46.0,32574,29,0.09


In [5]:
# Sketch the target column: ctr_gap = page's CTR minus its position tier's expected (median) CTR.
# This is computed only from columns already in hand -- no product scores, no trend fields.
tier_expected_ctr = lane_df.groupby("position_tier")["ctr"].transform("median")
lane_df["ctr_gap"] = lane_df["ctr"] - tier_expected_ctr

print("ctr_gap summary (negative = underperforming its tier):")
print(lane_df["ctr_gap"].describe())
print()

# quick sketch of the optional binary proxy layered on top of the score
lane_df["is_ctr_opportunity"] = (
    (lane_df["ctr_gap"] < lane_df.groupby("position_tier")["ctr_gap"].transform(
        lambda s: s.quantile(0.20)))
)
print(f"is_ctr_opportunity positives: {lane_df['is_ctr_opportunity'].sum():,} "
      f"of {len(lane_df):,} rows ({lane_df['is_ctr_opportunity'].mean():.1%})")

lane_df[unit_cols + ["ctr_gap", "is_ctr_opportunity"]].sort_values("ctr_gap").head(10)


ctr_gap summary (negative = underperforming its tier):
count    22006.000000
mean         0.105696
std          0.391143
min         -0.230000
25%         -0.070000
50%          0.000000
75%          0.180000
max         11.530000
Name: ctr_gap, dtype: float64

is_ctr_opportunity positives: 1,775 of 22,006 rows (8.1%)


,content_id,client_id,position_tier,avg_position,impressions_90d,clicks_90d,ctr,ctr_gap,is_ctr_opportunity
1598,content_8bd3cd8be92f,client_19581e27de,page_1,4.8,503,0,0.0,-0.23,True
27993,content_26d48a980581,client_f369cb89fc,page_1,4.6,1266,0,0.0,-0.23,True
16832,content_43cee7a2e67e,client_f369cb89fc,page_1,5.3,295,0,0.0,-0.23,True
16825,content_2a12375a7402,client_19581e27de,page_1,8.4,193,0,0.0,-0.23,True
3969,content_d3b4a729c0c2,client_19581e27de,page_1,8.3,210,0,0.0,-0.23,True
29193,content_fa0a7c87c208,client_8722616204,page_1,9.5,194,0,0.0,-0.23,True
29191,content_268547787415,client_f369cb89fc,page_1,6.6,116,0,0.0,-0.23,True
1618,content_97e778824a45,client_6208ef0f77,page_1,7.3,399,0,0.0,-0.23,True
16035,content_7dbdac24c5ac,client_f74efabef1,page_1,8.3,109,0,0.0,-0.23,True
22572,content_dd4364510947,client_f369cb89fc,page_1,5.1,230,0,0.0,-0.23,True


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single hard-coded CTR threshold (like "flag anything under 1%") doesn't work here because **expected CTR isn't constant — it depends heavily on position tier**, and the relationship isn't a clean straight line. From the w01 numbers: mean CTR runs from **1.48% at top_3** down to **0.65% at page_1**, **0.32% at striking**, **0.22% at page_3_5**, and **0.15% at deep**. A flat threshold either floods the queue with normal-but-lower-tier pages or misses real underperformers sitting in a high-value tier.

It gets messier once I add a second dimension: content type and intent also shift baseline CTR (informational vs. transactional queries don't click the same way even at the same position), so the "expected" value is really a small grid, not one number — exactly the kind of multi-signal, shifting pattern the framing skill says is worth reaching past a plain rule for. A simple **grouped baseline (median-per-tier, as I sketched above)** already beats one global threshold, and a model (even something as simple as a regression on position + tier + content type) can go further by learning the *shape* of the position-CTR curve and interaction effects I'd otherwise have to hand-tune per tier per content type.

That said, I'm not reaching for a complex model before earning it: my plan for the next few weeks is to (1) ship the grouped-median baseline first, since the lane guide explicitly wants a baseline to beat, then (2) test whether a simple model materially improves precision@50 over that baseline before claiming ML was worth it. If a two-line groupby beats a random forest, the honest answer is to use the groupby.


In [6]:
# Show the messiness directly: expected CTR isn't one number -- it varies by tier
# AND by content type, which is why a single threshold can't capture it.
tier_type_expected = (
    lane_df.assign(content_type=df.loc[lane_df.index, "content_type"])
    .groupby(["position_tier", "content_type"])["ctr"]
    .median()
    .unstack()
)
print("Expected (median) CTR by position tier x content type -- not a flat surface:")
print(tier_type_expected.round(2))


Expected (median) CTR by position tier x content type -- not a flat surface:
content_type   comparison article  feedly article  keyword article
position_tier                                                     
deep                          NaN            0.00             0.00
page_1                        0.0            0.32             0.23
page_3_5                      0.0            0.00             0.06
striking                      0.0            0.15             0.16
top_3                         0.0            1.93             0.19


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.